In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

: 

In [ ]:
class HybridRetriever:
    def __init__(self, document, alpha):
        self.document = document
        self.alpha = alpha
    
#----------------- BUILDING BM25 INDEX --------------#
        self.tokenized_docs = [doc.lower().split() for doc in document]
        self.bm25 = BM25Okapi(self.tokenized_docs)

#------------- BIENCODER ------------------------#
        self.embedding_model = (SentenceTransformer("all-MiniLM-L6-v2"))

        self.doc_embedding = (self.embedding_model.encode(
            document,
            normalize_embeddings=True,
            convert_to_numpy=True
        ))

#-------------- CROSS ENCODER -----------------------#
        self.cross_encoder = (CrossEncoder("cross-encoder/ms-macro-MiniLM-L6-V2"))

#-------------- NORMALIZE ----------------------------#

def _normalize(self, scores):
    scores = np.array(scores)

    if scores.max() == scores.min():
        return np.ones_like(scores)
    return (scores - scores.min()/ scores.max() - scores.min())

#----------------- HYBRID RETRIEVAL -----------------#
def hybrid_retrieval(self, query, top_k = 50):

    #-------BM25--------#
    query_tokens = (query.lower().split())
    bm25_scores = (self.bm25.get_scores(query_tokens))

    #---DENSE RETRIEVAL---#
    query_embedding = (self.embedding_model.encode(
        query,
        normalize_embeddings = True,
        convert_to_numpy = True
    ))

    dense_scores = np.dot(
        self.doc_embeddings,
        query_embedding

    )

    #----- SCORE NORMALIZATION ------#

    bm25_scores = (self._normalize(
        bm25_scores
    ))

    dense_scores = (self._normalize(
        dense_scores
    ))

    #=======WEIGHTED FUSION =========#

    hybrid_scores = (
        self.alpha * bm25_scores
        +
        (1-self.alpha) * dense_scores
    )

    ranked_indices = np.argsort(
        hybrid_scores
    )[::-1]

    results = []

    for idx in ranked_indices[:top_k]:
        results.append({
            "doc_id": idx,
            "document": self.documents[idx],
            "embedding": self.doc_embeddings[idx],
            "hybrid_score": float(
                hybrid_scores[idx]
            ),
            "bm25_scores": float(
                bm25_scores[idx]
            ),

            "dense_scores": float(
                dense_scores[idx]
            )

        })

        return results
    
    #=======================
    #MMR
    #=======================

    def mmr(self, query, retrieved_docs, top_k = 15, lambda_parameter = 0.7):
        query_embedding = (self.embedding_model.encode(query,
                           normalize_embeddings = True,
                           convert_to_numpy = True))
        
        doc_embedding = np.array([
            d["embedding"]
            for d in retrieved_docs
        ])

        query_similarities = np.dot(
            doc_embedding,
            query_embedding
        )

        selected = []
        remaining = list[range(len(retrieved_docs))]

        #===========================
        #FIRST DOCUMENT
        #===========================

        first_idx = np.argmax(query_similarities)
        selected.append(first_idx)
        selected.remove(first_idx)


        #===============================
        #GREEDY MMR
        #===============================

        while (
            len(selected) < top_k and remaining
        ):
            
            best_score = -np.inf
            best_doc = None

            for idx in remaining:
                relevance = (
                    query_similarities[idx]
                )

                redundancy = max(
                    np.dot(
                        doc_embedding[idx],
                        doc_embedding[s]
                    )
                    for s in selected
                )

                mmr_score = (
                    lambda_parameter * relevance
                    -
                    (1-lambda_parameter) * redundancy
                )

                if (mmr_score > best_doc):
                    best_score = mmr_score
                    best_doc = idx

                selected.append( best_doc)

                remaining.remove (best_doc)

                return[
                    retrieved_docs[i]
                    for i in selected
                ]
            
    def rerank(self, query, docs, top_k = 5):
        pairs = [
            (
                query,
                d["document"]
                for d in docs
            )
        ]

        scores = (
            self.cross_enocder.predict(
                pairs
            )
        )

        for doc, score in zip(
            docs,
            scores
        ):
            doc["cross_encoder_score"] = float(score)

        docs.sort(key = lambda x:
                  x["cross_encoder_score"],
                  reverse = True)
        return docs[:top_k]












In [ ]:
class BM25Retriver:
    def __init__(self, docu):
        self.docu = docu
        self.docu_token =[docu.lower().split()]

        self.bm25_calc = BM25Okapi(self.docu_token)

    def _normalize(score):
        scores_cal = np.array(score)
        if scores_cal.min() == scores_cal.max():
            return(np.ones_like(scores_cal))
        return(
            (score - scores_cal.min()/
             scores_cal.max() - scores_cal.min())
        )
    def search_25(self, query, top_k = 2):
        query_tok = (query.lower().split())
        self.embedding_model = (SentenceTransformer(
            "all-MiniLm-L6-V2"
        ))
        query_embed = (self.embedding_model.encode(
            query_tok,
            normalize_embeddings=True,
            convert_to_numpy=True

        ))
        bm25_scores = (self.bm25_calc.get_scores(
            query_tok

        ))
        bm25_scores = (
            self._normalize(bm25_scores)
        )

        
        


In [ ]:
class DenseRetrv:
    def __init__(self, docs, query):
        self.tokenize_docs = [docs.lower().split()]
        self.model_embed = BM25Okapi(self.tokenize_docs)

    def _normalize(scores):
        score_calc = np.array(scores)
        